In [ ]:
import pandas as pd

In [ ]:
path="/content/datasets"

In [ ]:
files={
    "Netflix":("netflix titles.csv","netflix credits.csv"),
    "Amazon Prime":("amazonprime titles.csv","amazonprime credits.csv"),
    "Disney+":("Disney+ titles.csv","Disney+ credits.csv"),
    "HBO":("HBO titles.csv","HBO credits.csv"),
    "Apple TV":("appletv titles.csv","appletv credits.csv"),
    "Paramount+":("Paramount+ titles.csv","Paramount+ credits.csv")
}

In [ ]:
title_list=[]
credit_list=[]
for ott,(title_file,credit_file) in files.items():
    temp=pd.read_csv(f"{path}/{title_file}")
    temp["available_on"]=ott
    title_list.append(temp)
    credit_list.append(pd.read_csv(f"{path}/{credit_file}"))

In [ ]:
titles=pd.concat(title_list,ignore_index=True)
credits=pd.concat(credit_list,ignore_index=True)

In [ ]:
platforms=titles.groupby("id")["available_on"].agg(lambda x:", ".join(sorted(set(x))))

In [ ]:
df=titles.drop(columns="available_on")
df=df.sort_values("imdb_votes",ascending=False,kind="stable")
df=df.groupby("id",as_index=False).first()
df["available_on"]=df["id"].map(platforms)

In [ ]:
credits=credits.drop_duplicates(subset=["id","person_id","role"])
credits=credits.dropna(subset=["name"])

In [ ]:
directors=credits[credits["role"]=="DIRECTOR"].groupby("id")["name"].agg(", ".join)
cast=credits[credits["role"]=="ACTOR"].groupby("id")["name"].agg(lambda x:", ".join(x.head(5)))

In [ ]:
df["director"]=df["id"].map(directors)
df["cast"]=df["id"].map(cast)

In [ ]:
df.to_csv(f"merged_ott_dataset.csv",index=False)

In [ ]:
print("Rows before merging:",len(titles))
print("Unique titles:",len(df))
print("On more than one platform:",df["available_on"].str.contains(",").sum())
print(df[["description","imdb_score","director","cast"]].isna().sum())

Rows before merging: 25246
Unique titles: 23357
On more than one platform: 1870
description     195
imdb_score     2557
director       6063
cast           2583
dtype: int64


In [ ]:
df.head()

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,available_on,director,cast
0,tm1,Star Wars,MOVIE,Princess Leia is captured and held hostage by ...,1977,PG,121,"['action', 'scifi', 'fantasy']",['US'],NaN,tt0076759,8.6,1378285.0,81.289,8.207,Disney+,George Lucas,"Mark Hamill, Harrison Ford, Carrie Fisher, Pet..."
1,tm10,The Matrix,MOVIE,"Set in the 22nd century, The Matrix tells the ...",1999,R,136,"['action', 'scifi']",['US'],NaN,tt0133093,8.7,1934137.0,76.460,8.200,HBO,"Lana Wachowski, Lilly Wachowski","Keanu Reeves, Laurence Fishburne, Carrie-Anne ..."
2,tm10000,Jury Duty,MOVIE,When jobless Tommy Collins discovers that sequ...,1995,PG-13,88,"['comedy', 'drama']",['US'],NaN,tt0113500,4.2,8331.0,8.846,4.500,HBO,John Fortenberry,"Pauly Shore, Tia Carrere, Stanley Tucci, Richa..."
3,tm100001,The Lucky Texan,MOVIE,"Jerry Mason, a young Texan, and Jake Benson, a...",1934,None,61,"['western', 'action', 'romance']",['US'],NaN,tt0025440,5.6,1284.0,3.703,4.900,"Amazon Prime, Paramount+",Robert N. Bradbury,"John Wayne, Barbara Sheldon, Lloyd Whitlock, G..."
4,tm1000022,Boonie Bears: The Wild Life,MOVIE,Bear brothers Briar and Bramble set off on an ...,2021,None,99,"['comedy', 'animation']",['CN'],NaN,tt11654032,4.5,164.0,9.294,4.900,Amazon Prime,Liang Ding,"Xiao Tan, Wei Zhang, Bingjun Zhang, Pei Liu"


In [ ]:
df.columns

Index(['id', 'title', 'type', 'description', 'release_year',
       'age_certification', 'runtime', 'genres', 'production_countries',
       'seasons', 'imdb_id', 'imdb_score', 'imdb_votes', 'tmdb_popularity',
       'tmdb_score', 'available_on', 'director', 'cast'],
      dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23357 entries, 0 to 23356
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    23357 non-null  object 
 1   title                 23357 non-null  object 
 2   type                  23357 non-null  object 
 3   description           23162 non-null  object 
 4   release_year          23357 non-null  int64  
 5   age_certification     11379 non-null  object 
 6   runtime               23357 non-null  int64  
 7   genres                23357 non-null  object 
 8   production_countries  23357 non-null  object 
 9   seasons               5623 non-null   float64
 10  imdb_id               21408 non-null  object 
 11  imdb_score            20800 non-null  float64
 12  imdb_votes            20740 non-null  float64
 13  tmdb_popularity       22637 non-null  float64
 14  tmdb_score            20357 non-null  float64
 15  available_on       

In [ ]:
df["description"]=df["description"].str.replace(r"\s+"," ",regex=True).str.strip()
df=df[df["description"].fillna("")!=""].copy()

In [ ]:
df["title"]=df["title"].str.replace(r"\s+"," ",regex=True).str.strip()

In [ ]:
#genres list to plain words
df["genres_clean"]=df["genres"].fillna("").str.replace(r"[\[\]',]","",regex=True).str.strip()

In [ ]:
df["director"]=df["director"].fillna("")
df["cast"]=df["cast"].fillna("")

In [ ]:
#join first and last name so each person is one word
def join_names(text):
    names=text.split(",")
    return " ".join(name.replace(" ","").lower() for name in names)

In [ ]:
df["director_clean"]=df["director"].apply(join_names)
df["cast_clean"]=df["cast"].apply(join_names)

In [ ]:
df["director_clean"]=df["director"].apply(join_names)
df["cast_clean"]=df["cast"].apply(join_names)

In [ ]:
df.to_csv(f"cleaned_ott_dataset.csv",index=False)

In [ ]:
print("Cleaned rows:",len(df))
print("Missing IMDb scores:",df["imdb_score"].isna().sum())
print("Titles without genres:",(df["genres_clean"]=="").sum())

Cleaned rows: 23162
Missing IMDb scores: 2415
Titles without genres: 335


In [ ]:
df[["title","genres_clean","director_clean","cast_clean"]].head()

,title,genres_clean,director_clean,cast_clean
0,Star Wars,action scifi fantasy,georgelucas,markhamill harrisonford carriefisher petercush...
1,The Matrix,action scifi,lanawachowski lillywachowski,keanureeves laurencefishburne carrie-annemoss ...
2,Jury Duty,comedy drama,johnfortenberry,paulyshore tiacarrere stanleytucci richardrieh...
3,The Lucky Texan,western action romance,robertn.bradbury,johnwayne barbarasheldon lloydwhitlock george'...
4,Boonie Bears: The Wild Life,comedy animation,liangding,xiaotan weizhang bingjunzhang peiliu


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 23162 entries, 0 to 23356
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    23162 non-null  object 
 1   title                 23162 non-null  object 
 2   type                  23162 non-null  object 
 3   description           23162 non-null  object 
 4   release_year          23162 non-null  int64  
 5   age_certification     11345 non-null  object 
 6   runtime               23162 non-null  int64  
 7   genres                23162 non-null  object 
 8   production_countries  23162 non-null  object 
 9   seasons               5565 non-null   float64
 10  imdb_id               21265 non-null  object 
 11  imdb_score            20747 non-null  float64
 12  imdb_votes            20689 non-null  float64
 13  tmdb_popularity       22564 non-null  float64
 14  tmdb_score            20319 non-null  float64
 15  available_on          23

In [ ]:
import re
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [ ]:
stemmer=SnowballStemmer("english")

In [ ]:
def clean_story(text):
    text=re.sub(r"[^a-z\s]"," ",text.lower())
    words=[stemmer.stem(word) for word in text.split() if word not in ENGLISH_STOP_WORDS]
    return " ".join(words)

In [ ]:
df["story"]=df["description"].apply(clean_story)

In [ ]:
weights={
    "story":0.5,
    "genres_clean":0.3,
    "cast_clean":0.1,
    "director_clean":0.1
}

In [ ]:
features=list(weights.keys())
df[features]=df[features].fillna("")

In [ ]:
print("Total weight:",round(sum(weights.values()),2))

Total weight: 1.0


In [ ]:
for col in features:
    print(col,"empty rows:",(df[col]=="").sum())

story empty rows: 2
genres_clean empty rows: 335
cast_clean empty rows: 2437
director_clean empty rows: 5901


In [ ]:
df[["title"]+features].head()

,title,story,genres_clean,cast_clean,director_clean
0,Star Wars,princess leia captur held hostag evil imperi f...,action scifi fantasy,markhamill harrisonford carriefisher petercush...,georgelucas
1,The Matrix,set nd centuri matrix tell stori comput hacker...,action scifi,keanureeves laurencefishburne carrie-annemoss ...,lanawachowski lillywachowski
2,Jury Duty,jobless tommi collin discov sequest juror earn...,comedy drama,paulyshore tiacarrere stanleytucci richardrieh...,johnfortenberry
3,The Lucky Texan,jerri mason young texan jake benson old ranche...,western action romance,johnwayne barbarasheldon lloydwhitlock george'...,robertn.bradbury
4,Boonie Bears: The Wild Life,bear brother briar brambl set adventur human f...,comedy animation,xiaotan weizhang bingjunzhang peiliu,liangding


In [ ]:
import numpy as np
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
vectorizers={}
matrices=[]

In [ ]:
#vectorize each feature and apply its weight
for col in features:
    tfidf=TfidfVectorizer(token_pattern=r"\S+")
    matrix=tfidf.fit_transform(df[col])
    matrices.append(matrix*np.sqrt(weights[col]))
    vectorizers[col]=tfidf
    print(col,matrix.shape)

story (23162, 32529)
genres_clean (23162, 19)
cast_clean (23162, 48973)
director_clean (23162, 12232)


In [ ]:
final_matrix=hstack(matrices).tocsr()
print("Final matrix:",final_matrix.shape)

Final matrix: (23162, 93753)


In [ ]:
#weighted similarity of one title with all titles
def get_similarity(idx):
    return (final_matrix@final_matrix[idx].T).toarray().ravel()

In [ ]:
df=df.reset_index(drop=True)

In [ ]:
#quick check
idx=df[df["title"]=="The Avengers"].index[0]
scores=get_similarity(idx)
top=[i for i in scores.argsort()[::-1] if i!=idx][:5]
for i in top:
    print(df["title"].iloc[i],df["release_year"].iloc[i],round(float(scores[i]),2))

Avengers: Age of Ultron 2015 0.54
Captain America: The Winter Soldier 2014 0.38
Captain America: Civil War 2016 0.36
Avengers: Infinity War 2018 0.35
Captain America: The First Avenger 2011 0.34


In [ ]:
#rating based on score and number of votes, unrated titles get average
avg=df["imdb_score"].mean()
m=df["imdb_votes"].median()
v=df["imdb_votes"].fillna(0)
df["rating"]=(v*df["imdb_score"].fillna(avg)+m*avg)/(v+m)

In [ ]:
def recommend(title,platform="all",content_type="both",year=None):
    matches=df[df["title"].str.lower()==title.strip().lower()]
    if year:
        matches=matches[matches["release_year"]==year]
    if matches.empty:
        print("Title not found")
        return

    #if many titles have same name, take the most voted one
    idx=matches["imdb_votes"].fillna(0).idxmax()
    print("Because you liked",df["title"][idx],f"({df['release_year'][idx]})\n")

    data=df.copy()
    data["similarity"]=get_similarity(idx)
    data=data.drop(idx)

    if platform.lower()!="all":
        data=data[data["available_on"].str.lower().str.contains(platform.lower(),regex=False)]
    if content_type.lower()!="both":
        data=data[data["type"].str.lower()==content_type.lower()]

    data=data[data["similarity"]>0].nlargest(50,"similarity")
    if data.empty:
        print("No results found")
        return

    data["final_score"]=0.8*data["similarity"]+0.2*data["rating"]/10
    top=data.nlargest(5,"final_score")
    return top[["title","release_year","type","imdb_score","available_on"]].reset_index(drop=True)

In [ ]:
recommend("The Avengers")

Because you liked The Avengers (2012)



,title,release_year,type,imdb_score,available_on
0,Avengers: Age of Ultron,2015,MOVIE,7.3,Disney+
1,Captain America: The Winter Soldier,2014,MOVIE,7.8,Disney+
2,Avengers: Infinity War,2018,MOVIE,8.4,Disney+
3,Captain America: Civil War,2016,MOVIE,7.8,Disney+
4,Avengers: Endgame,2019,MOVIE,8.4,Disney+


In [ ]:
recommend("Pirates of the Caribbean: The Curse of the Black Pearl","all","both")

Because you liked Pirates of the Caribbean: The Curse of the Black Pearl (2003)



,title,release_year,type,imdb_score,available_on
0,Pirates of the Caribbean: Dead Man's Chest,2006,MOVIE,7.3,Disney+
1,Pirates of the Caribbean: At World's End,2007,MOVIE,7.1,Disney+
2,Pirates of the Caribbean: On Stranger Tides,2011,MOVIE,6.6,Disney+
3,Pirates of the Caribbean: Dead Men Tell No Tales,2017,MOVIE,6.5,Disney+
4,The Lord of the Rings: The Return of the King,2003,MOVIE,9.0,"HBO, Netflix"


In [ ]:
from difflib import get_close_matches

In [ ]:
types=["movie","show","both"]
otts=["netflix","amazon prime","disney","hbo","apple","paramount","all"]
all_titles=df["title"].str.lower().unique()

In [ ]:
def ask_option(question,options):
    while True:
        answer=input(f"{question} ({', '.join(options)}): ").strip().lower()
        if answer in options:
            return answer
        print("Please choose from the given options")

def ask_title():
    while True:
        name=input("Enter a movie/show you liked: ").strip().lower()
        matches=df[df["title"].str.lower()==name]
        if not matches.empty:
            return matches.sort_values("imdb_votes",ascending=False)
        print("Title not found")
        close=get_close_matches(name,all_titles,n=5)
        if close:
            print("Did you mean:",", ".join(df[df["title"].str.lower()==c]["title"].iloc[0] for c in close))

matches=ask_title().drop_duplicates("release_year")
idx=matches.index[0]

#let user choose the year if many titles have same name
if len(matches)>1:
    for n,(_,row) in enumerate(matches.iterrows(),1):
        print(f"{n}. {row['title']} ({row['release_year']}) - {row['type']}")
    numbers=[str(n) for n in range(1,len(matches)+1)]
    choice=ask_option("Which one",numbers)
    idx=matches.index[int(choice)-1]

platform=ask_option("OTT",otts)
content_type=ask_option("Type",types)

result=recommend(df["title"][idx],platform,content_type,year=df["release_year"][idx])
if result is not None:
    display(result)

Enter a movie/show you liked: Pirates of the Caribbean: The Curse of the Black Pearl
OTT (netflix, amazon prime, disney, hbo, apple, paramount, all): all
Type (movie, show, both): both
Because you liked Pirates of the Caribbean: The Curse of the Black Pearl (2003)



,title,release_year,type,imdb_score,available_on
0,Pirates of the Caribbean: Dead Man's Chest,2006,MOVIE,7.3,Disney+
1,Pirates of the Caribbean: At World's End,2007,MOVIE,7.1,Disney+
2,Pirates of the Caribbean: On Stranger Tides,2011,MOVIE,6.6,Disney+
3,Pirates of the Caribbean: Dead Men Tell No Tales,2017,MOVIE,6.5,Disney+
4,The Lord of the Rings: The Return of the King,2003,MOVIE,9.0,"HBO, Netflix"


In [ ]:
#test with different kinds of titles
for title in ["The Dark Knight","Stranger Things","Toy Story","Friends"]:
    result=recommend(title)
    if result is not None:
        display(result)

Because you liked The Dark Knight (2008)



,title,release_year,type,imdb_score,available_on
0,The Dark Knight Rises,2012,MOVIE,8.4,HBO
1,Batman Begins,2005,MOVIE,8.2,HBO
2,The Batman,2022,MOVIE,7.8,HBO
3,The Departed,2006,MOVIE,8.5,HBO
4,"Batman: The Long Halloween, Part Two",2021,MOVIE,7.2,HBO


Because you liked Stranger Things (2016)



,title,release_year,type,imdb_score,available_on
0,Supernatural,2005,SHOW,8.4,Netflix
1,Wolf Pack,2023,SHOW,6.2,Paramount+
2,Truth Seekers,2020,SHOW,7.0,Amazon Prime
3,The Twilight Zone,1959,SHOW,9.1,Paramount+
4,Mortel,2019,SHOW,6.5,Netflix


Because you liked Toy Story (1995)



,title,release_year,type,imdb_score,available_on
0,Toy Story 2,1999,MOVIE,7.9,Disney+
1,Toy Story 3,2010,MOVIE,8.3,Disney+
2,Toy Story 4,2019,MOVIE,7.7,Disney+
3,A Bug's Life,1998,MOVIE,7.2,Disney+
4,Toy Story That Time Forgot,2014,MOVIE,7.1,Disney+


Because you liked Friends (1994)



,title,release_year,type,imdb_score,available_on
0,Boynton Beach Club,2006,MOVIE,6.5,Amazon Prime
1,Back for Good,2021,MOVIE,5.7,Amazon Prime
2,"Love, Victor",2020,SHOW,8.0,Disney+
3,Sex and the City,1998,SHOW,7.3,HBO
4,Breakfast at Tiffany's,1961,MOVIE,7.6,Paramount+


In [ ]:
engine={
    "data":df,
    "matrix":final_matrix,
    "vectorizers":vectorizers,
    "weights":weights
}

In [ ]:
import pickle

In [ ]:
with open(f"ott_recommendation_engine.pkl","wb") as f:
    pickle.dump(engine,f)